# Short-Term Memory

In [1]:
from agents import Agent, Runner, SQLiteSession, trace
from setup import bedrock_tool

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


In [2]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant comparing how healthy different foods are.
    If you answer, give a list of how healthy the foods are with a score from 1 to 10. Order by: healtiest food comes first.

    Example:
    Q: Compare X and Y
    A: X is healtier as Y.
    1) X: 8/10 - Very healthy but high in fructose
    2) Y: 3/10 - High in sugar and fat
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
)

## No Memory

In [3]:
with trace("Compare without Memory") as t:
    result = await Runner.run(nutrition_agent, "Which is healthier, bananas or lollipop?")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")


Output: Certainly! Here’s a comparison of the healthiness of bananas and lollipops:

1) **Bananas**: 9/10
   - **Nutritional Benefits**: High in essential nutrients such as potassium, vitamin B6, and vitamin C. They also provide dietary fiber, which is beneficial for digestion and heart health. Bananas are naturally low in fat and calories.

2) **Lollipop**: 1/10
   - **Nutritional Drawbacks**: Typically high in sugar and artificial ingredients with minimal to no nutritional benefits. They provide empty calories and can contribute to dental problems and weight gain.

In summary, bananas are much healthier than lollipops.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_cf463601fcdb42269b039a32bfd2815d


In [4]:
with trace("Add Apple - without Memory") as t:
    result = await Runner.run(nutrition_agent, "Add apples to the comparison")

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Sure, let's compare apples to a few other common foods based on their healthiness. I'll assign scores from 1 to 10 for each food.

1) Apples: 9/10 - High in fiber, vitamins (especially vitamin C), and antioxidants. They promote good digestion and heart health.
2) Broccoli: 9/10 - Rich in vitamins (A, C, K), fiber, and antioxidants. It supports immune function and bone health.
3) Blueberries: 8/10 - High in antioxidants, vitamins (C and K), and fiber. They support brain health and reduce inflammation.
4) Almonds: 7/10 - Good source of healthy fats, fiber, protein, and vitamin E. They support heart health and weight management.
5) Bananas: 7/10 - Good source of potassium, vitamin B6, and fiber. They support heart health and muscle function.
6) Spinach: 7/10 - High in iron, calcium, vitamins (A, C, K), and antioxidants. It supports bone health and reduces oxidative stress.
7) Whole Grains (e.g., Brown Rice, Quinoa): 7/10 - Provide fiber, B vitamins, and essential minerals. They su

## Short Term Memory

In [5]:
session = SQLiteSession("conversation_history")

In [6]:
result = await Runner.run(
    nutrition_agent, "Which is healthier, bananas or lollipop?", session=session
)
print(result.final_output)

A: Bananas are healthier than lollipops.

1) Bananas: 9/10 - High in potassium, fiber, and various vitamins and antioxidants.
2) Lollipops: 1/10 - High in sugar and additives with no nutritional value.


In [7]:
with trace("Compare with Memory") as t:
    result = await Runner.run(
        nutrition_agent, "Add apples to the comparison", session=session
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: A: Bananas are healthier than lollipops, and apples are the next healthiest.

1) Bananas: 9/10 - High in potassium, fiber, and various vitamins and antioxidants.
2) Apples: 8/10 - High in fiber, vitamin C, and various antioxidants.
3) Lollipops: 1/10 - High in sugar and additives with no nutritional value.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_b8e72b8838124634a47b7000fe44d031
